# DIOR Faster R-CNN Training

This notebook prepares DIOR in Google Colab and launches the ResNet-50 and MobileNet Faster R-CNN experiments. Model, dataset, and optimization logic live in the project source. These cells handle Colab setup and experiment execution.

Dataset archives are extracted to the Colab runtime. Checkpoints and logs are stored in Google Drive.

In [ ]:
# Load the project source into the Colab runtime.
!test -d /content/ecs174-dior/.git || git clone https://github.com/viveknarayana/ecs174-dior.git /content/ecs174-dior
%cd /content/ecs174-dior
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
# Pull the latest project updates before using the cloned source.
!git -C /content/ecs174-dior pull origin main


In [ ]:
# Download the DIOR archives and extract them into the directory structure expected by DIORDataset.
!pip -q install gdown matplotlib
!mkdir -p /content/dior_archives /content/DIOR
!gdown --folder 'https://drive.google.com/drive/folders/1UdlgHk49iu6WpcJ5467iT-UqNPpx__CC' -O /content/dior_archives

from pathlib import Path
from zipfile import ZipFile
archive_dir = Path('/content/dior_archives')
data_root = Path('/content/DIOR')
for archive in sorted(archive_dir.glob('*.zip')):
    print('Extracting', archive.name)
    with ZipFile(archive) as zf:
        zf.extractall(data_root)

assert (data_root / 'JPEGImages-trainval').is_dir(), 'JPEGImages-trainval folder not found after extraction'
assert (data_root / 'JPEGImages-test').is_dir(), 'JPEGImages-test folder not found after extraction'
assert (data_root / 'Annotations' / 'Horizontal Bounding Boxes').is_dir(), 'Annotations folder not found after extraction'
assert (data_root / 'ImageSets' / 'Main' / 'train.txt').is_file() or (data_root / 'Main' / 'train.txt').is_file(), 'train.txt split file not found after extraction'
print('DIOR folder layout looks ready:', data_root)

In [ ]:
# Keep checkpoints and logs on Drive because Colab runtime storage is temporary.
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_ROOT = '/content/drive/MyDrive/DIOR_checkpoints'
print('Checkpoint destination:', OUTPUT_ROOT)

## Smoke test

This one-epoch run uses two training images and one validation image. It exercises dataset loading, the model forward and backward passes, validation loss, and checkpoint writing.

The smoke-test checkpoint, log, and loss plot are written under `MyDrive/DIOR_checkpoints/smoke_test/resnet50/`.

In [ ]:
!python train.py --data-root /content/DIOR --backbone resnet50 --output-root {OUTPUT_ROOT}/smoke_test --tiny-test --tiny-train-images 2 --tiny-validation-images 1 --epochs 1 --batch-size 2

## Full experiments

Each full run trains on the complete training split and computes validation loss on the complete validation split for 10 epochs. The test split is not used in these training runs.

Both configurations use 640-pixel model resizing, batch size 2, SGD with learning rate 0.005, momentum 0.9, weight decay 0.0005, and StepLR decay after epoch 5 with gamma 0.1. Backbone-specific settings are defined in `train.py`, and shared defaults are in `config.py`. A training and validation loss plot is saved as `loss_curves.png` in each run directory.

In [ ]:
# ResNet-50
!python train.py --data-root /content/DIOR --backbone resnet50 --output-root {OUTPUT_ROOT} --epochs 10 --batch-size 2

In [ ]:
# MobileNetV3
!python train.py --data-root /content/DIOR --backbone mobilenet --output-root {OUTPUT_ROOT} --epochs 10 --batch-size 2

## Resume an interrupted run

After reconnecting to a runtime, rerun the repository, data, and Drive setup cells. Resume from the latest completed checkpoint for the same backbone. `--epochs 10` is the total target, so resuming from epoch 3 continues at epoch 4.

In [ ]:
# Resume ResNet-50 from its epoch 3 checkpoint.
!python train.py --data-root /content/DIOR --backbone resnet50 --output-root {OUTPUT_ROOT} --epochs 10 --batch-size 2 --resume /content/drive/MyDrive/DIOR_checkpoints/resnet50/checkpoint_epoch_03.pth